# 폐쇄망 RAG 지식 커버리지 — Colab 실행 노트북

본실험은 완료됨. 남은 GPU 작업은 두 가지다.

- **작업 A** 판정자 다수결 채점 (보류 2,152건 해소) — 1~2시간
- **작업 B** 범위 밖 확장 문항 25개 실행 — 25분

결과는 Google Drive 에 즉시 저장한다. Kaggle 에서 두 번 잃은 원인이 세션
종료였으므로, 이번에는 10분마다 자동으로 Drive 에 복사한다.

In [ ]:
# [1] Drive 마운트 · 경로 설정 · GPU 확인
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, json, time, urllib.request, glob, re, zipfile
from pathlib import Path

DRIVE  = Path('/content/drive/MyDrive/oos_rag')   # 결과 영속 저장 (Drive)
EXP    = Path('/content/exp')                      # 작업 폴더 (임시)
MODELS = Path('/content/ollama_models')            # 모델 17GB — Drive 무료 15GB 로는 부족
HFC    = Path('/content/hf_cache')
for d in (DRIVE, EXP, MODELS, HFC):
    d.mkdir(parents=True, exist_ok=True)
os.environ['OLLAMA_MODELS'] = str(MODELS)
os.environ['HF_HOME'] = str(HFC)

print(subprocess.run('nvidia-smi -L', shell=True, capture_output=True, text=True).stdout)
print(subprocess.run('df -h /content | tail -1', shell=True, capture_output=True, text=True).stdout)
print("DRIVE =", DRIVE)

Mounted at /content/drive
GPU 0: Tesla T4 (UUID: GPU-ece3ac62-4042-c3ca-4cb2-8c078a3dd2d9)

overlay         113G   47G   66G  42% /

DRIVE = /content/drive/MyDrive/oos_rag


In [ ]:
# [2] Ollama · 파이썬 패키지 설치
if shutil.which('ollama') is None:
    print("zstd 설치 중...")
    subprocess.run('sudo apt-get install -y zstd', shell=True, check=True)
    print("Ollama 설치 중...")
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
else:
    print("ollama 이미 설치됨")
subprocess.run('pip install -q rank-bm25 sentence-transformers', shell=True, check=True)
print("설치 완료")

zstd 설치 중...
Ollama 설치 중...
설치 완료


실패 원인 파악을 위해 `curl` 명령어를 직접 실행하여 자세한 오류 메시지를 확인합니다.

In [ ]:
# [3] Drive 의 zip 을 작업 폴더에 풀고, 이전 세션 결과를 복원
z = None
for hit in sorted(glob.glob('/content/drive/MyDrive/**/[Dd]ataset*.zip', recursive=True)):
    z = hit
    break
assert z, f"zip 을 못 찾았습니다: {[p.name for p in Path('/content/drive/MyDrive').iterdir()][:15]}"
print(f"압축: {z} ({os.path.getsize(z)/1024/1024:.0f}MB)")
zipfile.ZipFile(z).extractall(EXP)

# zip 안에 상위 폴더가 한 겹 더 있으면 끌어올린다
if not (EXP / 'question_final.jsonl').exists():
    inner = [p for p in EXP.iterdir() if p.is_dir() and (p / 'question_final.jsonl').exists()]
    if inner:
        for item in list(inner[0].iterdir()):
            shutil.move(str(item), str(EXP / item.name))
        inner[0].rmdir()
        print("상위 폴더 한 겹 제거:", inner[0].name)

# runs 가 별도 zip 으로 있으면 함께 푼다
for extra in ['ALL_RESULTS.zip', 'RESULTS_14B.zip', 'ALL_CLOSEDBOOK.zip', 'RESULTS_OOS.zip']:
    for hit in glob.glob(f'/content/drive/MyDrive/**/{extra}', recursive=True):
        zipfile.ZipFile(hit).extractall(EXP)
        print("추가 해제:", extra)
        break

# 이전 세션에서 Drive 에 저장해 둔 결과 복원 (judge_cache 재사용이 핵심)
for sub in ['grade_out', 'grade_oos', 'runs_oos', 'analysis_final']:
    if (DRIVE / sub).is_dir():
        shutil.copytree(DRIVE / sub, EXP / sub, dirs_exist_ok=True)
        print("Drive 에서 복원:", sub)

os.chdir(EXP)
print("\n작업 폴더:", os.getcwd())
print(sorted(os.listdir('.'))[:25])
print("runs:", len(glob.glob('runs/responses_*.jsonl')),
      "| closedbook:", len(glob.glob('runs_closedbook/responses_*.jsonl')))

압축: /content/drive/MyDrive/Dataset.zip (183MB)
Drive 에서 복원: grade_out

작업 폴더: /content/exp
['17_sample_questions.py', '19_check_questions.py', '22_coverage_engine.py', '23_grade.py', '24_build_index.py', '25_rag_run.py', '26_merge_slots.py', '27_orchestrate.py', '28_dup_audit.py', '29_status_content.py', '30_regen.py', '31_make_worksheet.py', '32_fill_worksheet.py', '33_rebuild_questions.py', '34_fix_spans.py', '35_scan_refs.py', '36_closedbook.py', '37_snapshot.py', '38_analyze.py', '39_sample_oos.py', 'ALL_CLOSEDBOOK.zip', 'ALL_RESULTS.zip', 'CLOSEDBOOK_ALL.zip', 'GRADE_OUT.zip', 'RESULTS_14B.zip']
runs: 69 | closedbook: 3


In [ ]:
# [4] 스크립트 최신본·필수 데이터 검증 — 여기서 막히면 진행 금지
checks = {'23_grade.py': 'run_batch', '25_rag_run.py': 'set_ollama_host',
          '27_orchestrate.py': 'ollama_url', '38_analyze.py': 'short_name'}
bad = [f for f, key in checks.items()
       if not Path(f).exists() or key not in Path(f).read_text(encoding='utf-8')]
for f in checks:
    print(f"  {'구버전/없음' if f in bad else '최신본'}  {f}")

need = ['question_final.jsonl', 'status_content.json', 'index/chunks.jsonl']
miss = [f for f in need if not Path(f).exists()]
print("\n변형 폴더:", sorted(d for d in os.listdir('.') if d.startswith('cov_')))
print("누락:", miss or "없음")

# 문항 파일과 응답의 qid 정합성 — 어긋나면 채점이 0건이 된다
slots = {json.loads(l)['qid'] for l in open('question_final.jsonl', encoding='utf-8')}
f0 = sorted(glob.glob('runs/responses_*.jsonl'))[0]
rid = {json.loads(l)['qid'] for l in open(f0, encoding='utf-8')}
print(f"qid 교집합 {len(slots & rid)} / 응답 {len(rid)}")

assert not bad and not miss and len(slots & rid) > 0.9 * len(rid), "*** 검증 실패 ***"
print("\n검증 통과")

  최신본  23_grade.py
  최신본  25_rag_run.py
  최신본  27_orchestrate.py
  최신본  38_analyze.py

변형 폴더: ['cov_core', 'cov_periph', 'cov_random', 'cov_vol']
누락: 없음
qid 교집합 97 / 응답 97

검증 통과


In [ ]:
# [5] Ollama 서버 기동 — GPU 장수에 맞춰 자동 (Colab 은 보통 1장)
subprocess.run(['pkill', '-f', 'ollama serve'], check=False)
time.sleep(3)
for sub in ['manifests', 'blobs']:
    os.makedirs(MODELS / sub, exist_ok=True)   # 폴더 부재로 기동 실패하는 것 방지

n_gpu = len([l for l in subprocess.run('nvidia-smi -L', shell=True,
             capture_output=True, text=True).stdout.splitlines() if l.strip()])
SERVERS = {g: f'127.0.0.1:{11434 + g}' for g in range(max(n_gpu, 1))}
BASE = dict(OLLAMA_KEEP_ALIVE='-1', OLLAMA_MAX_LOADED_MODELS='1',
            OLLAMA_NUM_PARALLEL='1', OLLAMA_CONTEXT_LENGTH='16384',
            OLLAMA_FLASH_ATTENTION='1',
            OLLAMA_MODELS=str(MODELS), HF_HOME=str(HFC))

def genv(gpu):
    e = os.environ.copy()
    e.update(BASE)
    e['CUDA_VISIBLE_DEVICES'] = str(gpu)
    e['OLLAMA_HOST'] = SERVERS[gpu]
    return e

for gpu, host in SERVERS.items():          # 순차 기동
    subprocess.Popen(['ollama', 'serve'],
                     stdout=open(f'/content/ollama_gpu{gpu}.log', 'w'),
                     stderr=subprocess.STDOUT, env=genv(gpu))
    for _ in range(90):
        try:
            urllib.request.urlopen(f'http://{host}/api/tags', timeout=2)
            print(f"GPU{gpu} 서버 기동 ({host})")
            break
        except Exception:
            time.sleep(1)
    else:
        print(open(f'/content/ollama_gpu{gpu}.log').read()[-1500:])
        raise RuntimeError(f"GPU{gpu} 미기동")

JUDGE_URLS = ','.join(f'http://{h}/api/generate' for h in SERVERS.values())
print(f"GPU {n_gpu}장 | 서버 {len(SERVERS)}개")

GPU0 서버 기동 (127.0.0.1:11434)
GPU 1장 | 서버 1개


In [ ]:
# [6] 모델 준비 — 필요한 것만 받는다
#     작업 A(채점) 이면 NEED = JUDGES,  작업 B(OOS) 면 NEED = EXPT
JUDGES = ['gemma2:9b', 'llama3.1:8b', 'mistral-nemo']       # 판정자(피실험 모델과 다른 계열)
EXPT = ['qwen3:8b', 'kamekichi128/qwen3-4b-instruct-2507:latest', 'qwen3:14b']

NEED = EXPT        # ← 작업 B 를 할 때 EXPT 로 바꾸고 이 셀만 다시 실행

have = subprocess.run(['ollama', 'list'], capture_output=True, text=True, env=genv(0)).stdout
for m in NEED:
    tags = [l.split()[0] for l in have.splitlines()[1:] if l.strip()]
    if m in tags or m + ':latest' in tags:
        print(f"{m} 이미 있음")
    else:
        print(f"{m} 내려받는 중...")
        subprocess.run(['ollama', 'pull', m], check=True, env=genv(0))

print(subprocess.run('du -sh /content/ollama_models; df -h /content | tail -1',
                     shell=True, capture_output=True, text=True).stdout)

qwen3:8b 내려받는 중...
kamekichi128/qwen3-4b-instruct-2507:latest 내려받는 중...
qwen3:14b 내려받는 중...
16G	/content/ollama_models
overlay         113G   67G   47G  59% /



In [ ]:
# [7] GPU 적재 게이트 — CPU 폴백이면 여기서 중단
probe = NEED[0]
subprocess.run(['ollama', 'run', probe, '안녕'], env=genv(0), capture_output=True)
ps = json.loads(urllib.request.urlopen(f'http://{SERVERS[0]}/api/ps', timeout=15).read())
assert ps.get('models'), "적재 실패"
for m in ps['models']:
    pct = m['size_vram'] / m['size'] * 100 if m['size'] else 0
    print(f"{m['name']}: {m['size'] / 1e9:.1f}GB, VRAM {pct:.1f}%")
    assert pct >= 99, "*** CPU 폴백 — 진행 금지 ***"
print("GPU 적재 정상")

qwen3:8b: 7.5GB, VRAM 100.0%
GPU 적재 정상


In [ ]:
# [8] Drive 자동 스냅샷 — 10분마다 결과를 Drive 로 복사
#     Colab 은 유휴 시 연결이 끊긴다. Drive 에 있어야 살아남는다.
import threading

_stop = {'v': False}

def _snap(dirs=('grade_out', 'grade_oos', 'runs_oos', 'analysis_main',
                'analysis_bal3', 'analysis_all'), every=600):
    while not _stop['v']:
        for _ in range(every):
            if _stop['v']:
                return
            time.sleep(1)
        for d in dirs:
            if os.path.isdir(d):
                try:
                    shutil.copytree(d, DRIVE / d, dirs_exist_ok=True)
                except Exception as e:
                    print(f"  [스냅샷 실패] {d}: {type(e).__name__}")
        print(f"[{time.strftime('%H:%M:%S')}] Drive 스냅샷", flush=True)

threading.Thread(target=_snap, daemon=True).start()
print("Drive 자동 스냅샷 시작 (10분 간격)")

Drive 자동 스냅샷 시작 (10분 간격)


---
## 작업 A — 판정자 다수결 채점

보류 2,152건(30.8%)을 해소한다. **정답률과 과신 오답률이 여기서 확정된다.**

`grade_out/judge_cache.json` 이 있으면 이미 판정한 것은 재호출하지 않는다.
중간에 끊겨도 200건마다 캐시가 저장되므로, 셀을 다시 실행하면 이어서 돌린다.

In [ ]:
# [9] 판정자 채점 (1~2시간) — 200건마다 진행률과 남은 시간이 찍힌다
cmd = ['python', '-u', '23_grade.py', 'runs', 'question_final.jsonl',
       '--status', 'status_content.json', '--closedbook', 'runs_closedbook',
       '--covdirs', 'cov_core,cov_periph,cov_random,cov_vol',
       '--judges', ','.join(JUDGES), '--judge-urls', JUDGE_URLS,
       '--out', 'grade_out']
print(' '.join(cmd), '\n')
subprocess.run(cmd)

python -u 23_grade.py runs question_final.jsonl --status status_content.json --closedbook runs_closedbook --covdirs cov_core,cov_periph,cov_random,cov_vol --judges gemma2:9b,llama3.1:8b,mistral-nemo --judge-urls http://127.0.0.1:11434/api/generate --out grade_out 

[17:14:21] Drive 스냅샷
[17:24:22] Drive 스냅샷


CompletedProcess(args=['python', '-u', '23_grade.py', 'runs', 'question_final.jsonl', '--status', 'status_content.json', '--closedbook', 'runs_closedbook', '--covdirs', 'cov_core,cov_periph,cov_random,cov_vol', '--judges', 'gemma2:9b,llama3.1:8b,mistral-nemo', '--judge-urls', 'http://127.0.0.1:11434/api/generate', '--out', 'grade_out'], returncode=0)

In [ ]:
# [10] 채점 결과 확인 + Drive 저장
print(Path('grade_out/grade_report.md').read_text(encoding='utf-8')[:600])
shutil.copytree('grade_out', DRIVE / 'grade_out', dirs_exist_ok=True)
print("\nDrive 저장 완료:", sorted(os.listdir(DRIVE)))

# 채점 결과

- 응답 6984개 / 판정보류 41 / 생성실패 0 / 확신도 파싱실패 128
- 판정자 호출 1158회 (캐시 적중 0회)

확신도 파싱실패에는 모델이 '<높음|중간|낮음|모름>' 템플릿을 그대로
출력한 경우가 포함된다. 이를 '높음' 으로 세면 분포가 왜곡되므로
따로 집계하고 캘리브레이션에서 제외한다.

## 1. 조건별 상태 분포

| 모델 | 조건 | 커버리지 | 문자% | 정답 | 기권 | **과신오답** | 신중오답 | 부분 | 보류 | 정확도 | 과신오답률 |
|---|---|---|---|---|---|---|---|---|---|---|---|
| kamekichi128/qwen3-4b-instruct-2507:latest | cov100_vol030_doc | 100.0 | 43.0 | 45 | 23 | **26** | 1 | 1 | 1 | 47% | 27% |
| kamekichi128/qwen3-4b-instruct-2507:latest | cov100_vol040_doc | 100.0 | 43.0 | 45 | 23 | **26** | 1 | 1 | 1 | 47% | 27% |
| kamekichi128/qwen3-4b

Drive 저장 완료: ['analysis_all', 'analysis_bal3', 'analysis_main', 'grade_out']


---
## 작업 B — 범위 밖 확장 문항 25개 실행

4.6절의 클러스터를 17 → 42로 늘린다. 분량 축 6조건 × 3모델 = 약 25분.

**셀 [6] 의 `NEED` 를 `EXPT` 로 바꾸고 그 셀만 다시 실행**한 뒤 아래를 돌린다.

In [ ]:
# [11] 범위 밖 확장 실행 전 점검
QFILE = 'question_oos_extra_filled.jsonl'
assert os.path.exists(QFILE), f"{QFILE} 이 없습니다"
slots = [json.loads(l) for l in open(QFILE, encoding='utf-8')]
n_w = sum(1 for s in slots if (s.get('question_ko') or '').strip())
n_oos = sum(1 for s in slots if not s.get('answerable', True))
print(f"확장 문항 {len(slots)}개 | 작성 {n_w} | 범위밖 {n_oos}")
assert n_w == len(slots) and n_oos == len(slots), "미작성 또는 범위밖이 아닌 문항이 있습니다"

# 문항이 바뀌면 질의 캐시가 덮어써진다. 본 문항용을 백업.
if os.path.exists('index/query_cache.npz'):
    shutil.copy('index/query_cache.npz', 'index/query_cache_main.npz.bak')
    print("본 문항 질의 캐시 백업")

확장 문항 25개 | 작성 25 | 범위밖 25


In [ ]:
# [12] 범위 밖 확장 본실행 — 모델별 순차 (약 25분)
t0 = time.time()
for i, m in enumerate(EXPT):
    gpu = i % len(SERVERS)
    print(f"\n{'=' * 60}\n[{i + 1}/{len(EXPT)}] {m} @ GPU{gpu}\n{'=' * 60}", flush=True)
    subprocess.run(['python', '-u', '27_orchestrate.py', 'index', QFILE,
                    '--cov-dirs', 'cov_vol', '--covs', '100', '--models', m,
                    '--k', '5', '--num-ctx', '16384', '--timeout', '120',
                    '--run-plan', 'run_plan.json', '--out', 'runs_oos',
                    '--ollama-url', f'http://{SERVERS[gpu]}'])
    print(f"누적 {(time.time() - t0) / 60:.0f}분")

print("\n응답 파일:", len(glob.glob('runs_oos/responses_*.jsonl')))
shutil.copytree('runs_oos', DRIVE / 'runs_oos', dirs_exist_ok=True)
print("Drive 저장 완료")


[1/3] qwen3:8b @ GPU0
[21:47:27] Drive 스냅샷
누적 10분

[2/3] kamekichi128/qwen3-4b-instruct-2507:latest @ GPU0
누적 16분

[3/3] qwen3:14b @ GPU0
[21:57:29] Drive 스냅샷
[22:07:30] Drive 스냅샷
누적 33분

응답 파일: 15
Drive 저장 완료


In [ ]:
# [13] 확장 문항 채점 (판정자 불필요 — 범위밖은 기권 여부만 본다)
main = [json.loads(l) for l in open('question_final.jsonl', encoding='utf-8')]
extra = [json.loads(l) for l in open('question_oos_extra_filled.jsonl', encoding='utf-8')]
with open('question_all.jsonl', 'w', encoding='utf-8') as f:
    for s in main + extra:
        f.write(json.dumps(s, ensure_ascii=False) + '\n')
print(f"합계 {len(main) + len(extra)}문항 | "
      f"범위밖 {sum(1 for s in main + extra if not s.get('answerable', True))}")

subprocess.run(['python', '23_grade.py', 'runs_oos', 'question_all.jsonl',
                '--covdirs', 'cov_vol', '--out', 'grade_oos'])

with open('grade_out/graded_all.jsonl', 'w', encoding='utf-8') as out:
    for f in ['grade_out/graded.jsonl', 'grade_oos/graded.jsonl']:
        if os.path.exists(f):
            out.write(open(f, encoding='utf-8').read())
print("병합 완료 → grade_out/graded_all.jsonl")

합계 122문항 | 범위밖 42
병합 완료 → grade_out/graded_all.jsonl


---
## 최종 분석

세 명세를 각각 뽑는다.

- `analysis_main` 8B·4B 균형 표본 — **논문 본문의 회귀표**
- `analysis_bal3` 세 모델 공통 조건 — 강건성 검증
- `analysis_all` 전체 — 기술통계·McNemar·폐쇄북·그림

In [ ]:
# [14] 분석 3종 — CPU 작업
G = 'grade_out/graded_all.jsonl' if os.path.exists('grade_out/graded_all.jsonl') \
    else 'grade_out/graded.jsonl'
COV = 'cov_core,cov_periph,cov_random,cov_vol'
print("입력:", G, "\n")

for extra_args, out in [(['--models', '8b,4b'], 'analysis_main'),
                        (['--balanced'], 'analysis_bal3'),
                        ([], 'analysis_all')]:
    print(f"--- {out} ---")
    subprocess.run(['python', '38_analyze.py', G, '--covdirs', COV,
                    *extra_args, '--out', out])

입력: grade_out/graded_all.jsonl 

--- analysis_main ---
--- analysis_bal3 ---
--- analysis_all ---


In [ ]:
# [15] 최종 저장 — Drive 와 zip 둘 다
_stop['v'] = True                       # 스냅샷 중지
for d in ['grade_out', 'grade_oos', 'runs_oos',
          'analysis_main', 'analysis_bal3', 'analysis_all']:
    if os.path.isdir(d):
        shutil.copytree(d, DRIVE / d, dirs_exist_ok=True)
        print("Drive 저장:", d)

for d in ['analysis_main', 'analysis_bal3', 'analysis_all']:
    if os.path.isdir(d):
        shutil.make_archive(str(DRIVE / d), 'zip', '.', d)

print("\nDrive 내용:", sorted(os.listdir(DRIVE)))
print("\n--- analysis_main 요약 ---")
print(Path('analysis_main/analysis_report.md').read_text(encoding='utf-8')[:700])

Drive 저장: grade_out
Drive 저장: grade_oos
Drive 저장: runs_oos
Drive 저장: analysis_main
Drive 저장: analysis_bal3
Drive 저장: analysis_all

Drive 내용: ['analysis_all', 'analysis_all.zip', 'analysis_bal3', 'analysis_bal3.zip', 'analysis_main', 'analysis_main.zip', 'grade_oos', 'grade_out', 'runs_oos']

--- analysis_main 요약 ---
# 통계 분석

- 응답 6,264 (RAG 6,070 / 폐쇄북 194)
- 문항 122 · 조건 30 · 모델 2
- 판정보류 38 (0.6%)

구간은 Wilson 95%. 회귀 표준오차는 문항 클러스터 강건.
모델이 k 종이면 더미를 k-1 개 넣는다(기준: Qwen3-4B-Inst).

## 1. 조건별 비율 (95% 신뢰구간)

정확도는 **답변한 것 중** 정답률이다(기권 제외). 괄호 안은 사건/표본.

| 모델 | 조건 | 커버 | 문자% | 기권 | 과신오답 | 정확(답변중) | 인용정확 | OOS기권 |
|---|---|---|---|---|---|---|---|---|
| kamekichi128/qwen3-4b- | cov100_vol030_doc | 100 | 43.0 | 37.7% [30–47] (46/122) | 23.1% [17–31] (28/121) | 61.3% [50–72] (46/75) | 87.5% [78–93] (63/72) | 90.5% [78–96] (38/42) |
| kamekichi128/qwen3-4b- | cov100_vol040_doc | 100 | 43.0 | 37.7% [30–47] (46/122) | 23.1% [17–31] (28/121) | 61.3% [50–72] (46/75) | 87.5% [78–93] (63/72) | 90.5% [7